# 01 — The frame index, and deciding a frame's type from its pixels

Build steps 1 and 1.5 (`DECISIONS` D16). Two questions:

1. Can we read a frame and split it into CFA sub-planes without ever debayering (D4)?
2. Can we decide what a frame *is* from its own pixels, given that neither the folder
   nor `IMAGETYP` is trustworthy (D18)?

**What this notebook produces.** Every file this build step contributed to `results/`
is generated by a cell below, so the route from the archive to the number is readable
end to end:

| file | cell | cost |
|---|---|---|
| `frame_index.csv` | *Acquisition*, guarded — one row per frame in the archive | ~107 min over SMB |
| `archive_census.csv` | *Census* — the archive summarised by folder, type, gain, exposure | seconds |
| `ladder_census.csv` | *Census* — the NGC7000 grid, one row per rung | seconds |
| `unreadable_frames.csv` | *Census* — the frames that would not open | seconds |

Everything except the acquisition cell reads the committed `frame_index.csv` and runs in
seconds, so this notebook can be re-read and re-run without touching Z: and without
disturbing a frozen archive (D19).

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import cfa, fits as F

pd.set_option("display.width", 200)

RESULTS = pathlib.Path("..") / "results"
INDEX = RESULTS / "frame_index.csv"
ARCHIVE_ROOTS = [pathlib.Path(r"Z:\pix\_astro\raw\_by_type") / t
                 for t in ("bias", "dark", "flat", "light")]


def observing_night(date_obs):
    """The night a frame belongs to, not the calendar date it was written on.

    A session runs through midnight, so the boundary is local noon: subtracting
    12 h puts a whole night onto one date.  This matters more than it looks --
    on calendar dates the NGC7000 grid appears to span four nights per gain when
    it spans two (D28), and the dawn-clipping analysis loses its ordering.
    """
    t = pd.to_datetime(date_obs, format="mixed", errors="coerce")
    return (t - pd.Timedelta(hours=12)).dt.strftime("%Y-%m-%d")


def load_index():
    """Read the index, with the few derived columns every cell below wants.

    Safe to run while a refresh is in progress.  `refresh_index` writes a temp
    file and `os.replace`s it, which is atomic, so a reader sees either the
    previous complete file or the new one -- never a half-written one.  And
    rows are never removed (D19), so a partial index is simply a shorter one.
    Rows arrive in path order, which means bias, then dark, then flat, then
    light: a run still in progress has no lights yet.
    """
    idx = pd.read_csv(INDEX)
    idx["root"] = idx.path.str.extract(r"_by_type\\([^\\]+)", expand=False)
    idx["night"] = observing_night(idx.date_obs)
    # recompute rather than trusting the stored column, so the notebook is not
    # hostage to how csv round-tripped True/False/blank
    idx["agrees"] = idx.declared_type.str.lower() == idx.measured_type
    ok = idx[idx.status == "ok"].copy()
    print(f"{len(idx)} rows, {len(ok)} readable, "
          f"indexed {idx.indexed_at.min()} .. {idx.indexed_at.max()}")
    print("by folder:", dict(ok.root.value_counts()))
    return idx, ok


def reclassify(ok):
    """Re-derive measured_type from the stored feature columns.

    Every input `classify` takes is a column in the index, so a change to the
    classifier can be replayed over the archive without re-reading a single
    frame.  That is worth knowing: it is what makes the thresholds revisable.
    """
    return ok.apply(lambda r: F.classify(r, r.exptime), axis=1)

## Acquisition — how `frame_index.csv` was built

The one expensive cell in this notebook, and the only one that touches Z:. It is guarded
off by default: re-running it does **not** reproduce the committed snapshot, it produces a
*new* one, and if a frame has moved since it should. That is D19 working as intended, not a
failure — each row carries the `indexed_at` stamp it was measured under, and every file in
`results/` records the snapshot it used.

What the run does, and why it is affordable:

- **walks four type folders, not their parent.** `_by_type` also contains `_canon`, a
  retired camera with its own `bias/dark/flat/light` beneath it, and pointing one level
  higher would pull foreign frames into exactly the buckets where they would look plausible.
  `walk()` prunes it, and `refresh_index` independently marks any frame whose `INSTRUME` is
  not this rig — two guards, deliberately redundant (D26).
- **reads six 32-row blocks per frame, not the frame.** 0.25 s against 2.7 s, which is the
  difference between a 40-minute pass and an 11-hour one, and classification wants
  statistics rather than pixels.
- **is incremental and resumable.** A row is re-scanned only when `(size, mtime)` moves, and
  the file is rewritten every 200 frames, so an interrupted run resumes rather than restarts.

The snapshot this notebook reads below was taken on **2026-08-27T13:24:22**: 15,102 rows,
106.8 minutes, ~0.42 s per frame over SMB, 15,090 readable and 12 not.

In [ ]:
# Guarded on purpose: ~107 minutes over SMB, and it writes a new snapshot.
# Flip to True only between analyses, never during one (D19).
REACQUIRE = False

if REACQUIRE:
    rows = F.refresh_index(ARCHIVE_ROOTS, str(INDEX))
    print(f"{len(rows)} rows -> {INDEX}")
else:
    print(f"skipped; reading the committed snapshot at {INDEX}")

## Why classification reads row-blocks, not frames

The archive is ~15,000 frames of 16.6 MB on a network drive. Timed on this rig:

| operation | cost |
|---|---|
| open + read header | 0.09 s |
| open + read 32 rows via `hdu.section` | 0.16 s |
| open + read all 16.6 MB | 2.7 s |

Reading whole frames is an 11-hour pass; sampling is well under one. Nothing is lost,
because classification wants *statistics* and statistics converge long before you have
read four million pixels.

The blocks are **contiguous** rather than strided, and that is not an accident. Two of the
features are spatial — whether a bright pixel has a bright neighbour — and row striding
destroys exactly that. They are **spread down the frame** so that vignetting and amp glow,
both corner-weighted, are sampled rather than missed.

In [ ]:
idx, ok = load_index()

# a light if the index has reached them, otherwise whatever it has -- the point
# of this cell is the read geometry, not the frame
lights = ok[ok.measured_type == "light"]
path = lights.path.iloc[0] if len(lights) else ok.path.iloc[0]
if not len(lights):
    print("no lights indexed yet; showing", ok.measured_type.iloc[0], "instead\n")

blocks, header = F.sample_blocks(path)
print(path)
print(f"{len(blocks)} blocks of {blocks[0].shape}, dtype {blocks[0].dtype}")
print(f"{sum(b.nbytes for b in blocks) / 1e6:.2f} MB read, "
      f"of {header['NAXIS1'] * header['NAXIS2'] * 2 / 1e6:.1f} MB in the frame")

planes = cfa.split(blocks[0])
{name: (p.shape, float(np.median(p))) for name, p in planes.items()}

## The 12-bit ADC, seen directly

`FINDINGS` records this as a *suspicion*: the ASI585's ADC is 12-bit, ASIAIR stores 16-bit
FITS, and the values are believed to be bit-shifted x16. If that is true then every stored
value is an exact multiple of 16 — the low four bits are padding, not measurement.

That is a testable claim about the pixels, so we test it rather than believe it.

In [ ]:
sample = np.concatenate([b.ravel() for b in blocks])
frac16 = float(np.mean(sample % 16 == 0))
print(f"fraction of values that are exact multiples of 16: {frac16:.6f}")
print(f"distinct low-4-bit values present: {sorted(set((sample % 16).tolist()))}")
print(f"max value seen: {sample.max()}  =  {sample.max() // 16} x 16")

**Consequence.** One count in the file is not one count of the ADC — it is 1/16 of one, and
it can never be occupied. Any e-/ADU figure must state which ADU it means. The header's
`EGAIN` is in *12-bit* ADU, so using it against 16-bit file values inflates electron counts
16x. This is the unit trap that gates build step 3.

A second consequence is less obvious and more dangerous: the quantiser is coarse relative to
the read noise at low gain. If sigma is comparable to one step, a robust spread estimated
from a single frame measures the ADC rather than the sensor. We check that below on the
index, where every gain is represented.

## The classifier

Four decisions, in an order chosen so the cheapest and most trustworthy evidence goes first.

1. **Bias** — the shortest exposure the camera takes. Exposure is a *capture setting*, which
   D18 leaves trusted, so this is settled without a pixel argument.
2. **Clipped** — saturated across the frame. This is the one branch that is an *inference*
   rather than a measurement, and it is marked as such: every feature is degenerate (level
   pinned to full scale, sigma and clump zero), so there is nothing left to measure. The
   fallback is exposure — every flat in this archive is 1–3 s, so a clipped long exposure is
   a light that ran into dawn, not a flat. Saturation itself stays recorded in `sat_frac`,
   a *quality* attribute orthogonal to what the frame is (D25).
3. **Flat** — level an order of magnitude above the pedestal. Flats sit at tens of thousands
   of ADU; darks and lights sit within a few hundred of the pedestal. No overlap.
4. **Dark vs light** — the hard case, since at equal exposure they share a pedestal and a
   noise floor. What separates them is *shape*, not level: a star is a PSF spread over
   several pixels, a hot pixel is one defective site.

The dark/light feature is therefore connectedness. Among pixels above `median + 5 sigma`,
what fraction have an above-threshold neighbour? Measured **inside a sub-plane**, where
neighbouring samples share a colour filter and the same optical scale — on the mosaic they
do not, and the comparison would be meaningless.

Horizontal and vertical connectedness are counted separately and the **weaker** one is used.
A hot *column* — an ordinary CMOS defect — is vertically connected and would otherwise read
as a star. A real PSF is connected both ways.

In [ ]:
idx, ok = load_index()

FEATURES = ["level", "sigma", "tail_frac", "clump_frac", "clump_h", "clump_v"]
ok.groupby("measured_type")[FEATURES].median()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for t, g in ok.groupby("measured_type"):
    ax.scatter(g.level.clip(1, None), g.clump_frac, s=6, alpha=.4, label=t)
ax.set_xscale("log")
ax.axvline(F.FLAT_MIN_LEVEL * F.FULL_SCALE, ls="--", c="k", lw=.8)
ax.axhline(F.LIGHT_MIN_CLUMP, ls="--", c="k", lw=.8)
ax.set_xlabel("level  (median across CFA planes, file ADU)")
ax.set_ylabel("clump fraction  (weaker axis)")
ax.set_title("the two cuts that do the work")
ax.legend()
plt.show()

## Does the label agree with the pixels?

D18 predicted disagreement: some flats and darks were captured under a Light subframe type.
The index quantifies it. Disagreements are a **finding and a cleanup work-list** (D20), not
a hazard — nothing downstream reads `IMAGETYP`.

In [ ]:
print(pd.crosstab(ok.declared_type, ok.measured_type, margins=True))
print()
print(pd.crosstab(ok.root, ok.measured_type, margins=True))

In [ ]:
# only frames that carry a label can disagree with one
labelled = ok[ok.declared_type.notna()]
disagree = labelled[~labelled.agrees]
print(f"{len(disagree)} of {len(labelled)} labelled frames disagree with their pixels")
disagree.groupby(["declared_type", "measured_type"]).size()

## Quantisation vs read noise, across the gain axis

The claim to test: at low gain the single-frame robust sigma collapses onto multiples of
1.4826 x 16 = 23.72, because MAD lands on exactly one ADC step. Where that happens, the
number is a property of the quantiser and says nothing about the sensor.

In [ ]:
cal = ok[ok.measured_type.isin(["bias", "dark"])]
print(cal.groupby("gain").sigma.describe()[["count", "min", "50%", "max"]])
print()
print("quantisation step in sigma units: 1.4826 * 16 =", 1.4826 * 16)

## The NGC7000 ladder (D7)

The primary validation dataset. Six sub-exposure rungs, each totalling exactly 9600 s,
interleaved across four nights. Before trusting that description, confirm it from the index:
the counts, the total integration per rung, the gain, and the achieved temperature.

In [ ]:
lad = ok[ok.path.str.contains("NGC7000_tests", regex=False)]
print("frames:", len(lad))
print(pd.crosstab([lad.gain, lad.exptime], lad.measured_type, margins=True))
print()
print(lad.groupby(["gain", "exptime"]).agg(n=("exptime", "size"), total_s=("exptime", "sum")))

In [ ]:
# `night` is the observing night from the setup cell, so a session that runs
# through midnight counts once.  On calendar dates this table reads four nights
# per gain and contradicts D28.
print(pd.crosstab([lad.gain, lad.exptime], lad.night, margins=True))
print()
print("achieved temperature:", sorted(lad.ccd_temp.dropna().unique()))

## Calibration matching (D9, D32)

Which darks exist for the grid's rungs? D9 suspected two gaps — 15 s darks at gain 50 only,
and 240 s darks at gain 50 / −20 °C — and made a re-shoot conditional on the index
confirming them. A confirmed gap is a re-shoot, never a silent substitution.

The check runs at **both** gains, because D28 found the grid spans gain 50 and 252 rather
than 252 alone. It is also the cheapest kind of question this project asks: one `groupby`
over a committed CSV, no frame re-read.

In [ ]:
dk = ok[ok.measured_type == "dark"]
print(pd.crosstab([dk.gain, dk.exptime], dk.ccd_temp.round(0), margins=True))

In [ ]:
LADDER_EXP = [15.0, 30.0, 60.0, 120.0, 240.0, 480.0]

cov = []
for gain in (50.0, 252.0):
    m = dk[(dk.gain == gain) & (dk.ccd_temp.between(-10.6, -9.4))]
    avail = m.groupby("exptime").size()
    cov.append(pd.Series({e: int(avail.get(e, 0)) for e in LADDER_EXP},
                         name=f"gain {gain:.0f} @ -10C"))
coverage = pd.DataFrame(cov)
print(coverage)
print()
print("rungs with no matching dark:",
      [(r, e) for r in coverage.index for e in LADDER_EXP if coverage.loc[r, e] == 0]
      or "none -- D9's suspected gaps do not exist (D32)")

## Census — the three summary tables

`frame_index.csv` is one row per frame and too long to read. These three cells reduce it to
the tables the documents actually cite, and write them to `results/`. All three are pure
functions of the index: no frame is re-read, so they cost seconds and can be re-run after any
threshold change (`reclassify`, D25).

Each carries the `index_snapshot` it was built from, which is what makes a number in
`FINDINGS` traceable back to the frames behind it (D19).

In [ ]:
SNAPSHOT = idx.indexed_at.max()


def archive_census(ok, snapshot):
    """The archive, one row per (folder, measured type, gain, exposure).

    Grouping by *measured* type rather than the label is the whole point: this
    is what shows a dark folder holding two gains and two temperatures (D9), and
    what makes the mislabel count a work-list rather than a worry (D20).
    """
    rows = []
    for (root, mt, gain, exp), d in ok.groupby(
            ["root", "measured_type", "gain", "exptime"], dropna=False):
        rows.append({
            "index_snapshot": snapshot, "root": root, "measured_type": mt,
            "gain": gain, "exptime": exp, "n_frames": len(d),
            "ccd_temp_min": d.ccd_temp.min(), "ccd_temp_max": d.ccd_temp.max(),
            "nights": d.night.nunique(),
            "level_med": d.level.median(), "sigma_med": d.sigma.median(),
            "n_clipped": int((d.sat_frac >= 0.5).sum()),
            "n_mislabelled": int((d.type_agrees == False).sum()),
        })
    return pd.DataFrame(rows)


census = archive_census(ok, SNAPSHOT)
census.to_csv(RESULTS / "archive_census.csv", index=False)
print(f"{len(census)} rows -> archive_census.csv")
census.head(10)

In [ ]:
def ladder_census(ok, snapshot):
    """The NGC7000 grid, one row per rung.

    `total_s` is the column D28 turns on: equal integration per rung is what
    makes "whichever rung stacks best is the optimum" a valid reading, and it
    holds exactly within a gain (448x15 = 224x30 = ... = 6720 s) while the two
    gains differ.  `nights` is listed rather than counted, because *which*
    nights matters -- the two gains share none, so gain is confounded with sky.
    """
    lad = ok[ok.path.str.contains("NGC7000_tests", regex=False)]
    rows = []
    for (gain, exp), d in lad.groupby(["gain", "exptime"]):
        rows.append({
            "index_snapshot": snapshot, "gain": gain, "exptime": exp,
            "n_frames": len(d), "total_s": len(d) * exp,
            "n_clipped": int((d.sat_frac >= 0.5).sum()),
            "nights": "|".join(sorted(d.night.dropna().unique())),
            "ccd_temp_min": d.ccd_temp.min(), "ccd_temp_max": d.ccd_temp.max(),
            "sky_level_med": d.level.median(),
        })
    return pd.DataFrame(rows)


ladder = ladder_census(ok, SNAPSHOT)
ladder.to_csv(RESULTS / "ladder_census.csv", index=False)
print(f"{len(ladder)} rows -> ladder_census.csv")
ladder

In [ ]:
# Built from the full index, not `ok`: an unreadable frame is by definition not
# status "ok", and it must stay visible rather than be filtered out of existence.
unreadable = idx[idx.status.str.startswith("unreadable", na=False)]
unreadable[["path", "size", "status", "indexed_at"]].to_csv(
    RESULTS / "unreadable_frames.csv", index=False)
print(f"{len(unreadable)} unreadable -> unreadable_frames.csv")
print("sizes:", sorted(unreadable["size"].unique()))
unreadable[["path", "size", "status"]]

**What the census settled.** All 12 unreadable files are exactly 0 bytes and each is the last
or near-last frame of its run by sequence number — interrupted writes at session end, not
corruption (D29). Nothing is deleted: under D20 the archive does not change until the model
is validated, so they stay marked and fenced out of `status == "ok"`.

---

## Where this leaves the build

Steps 1 and 1.5 are done. The index answers three questions that were open when it started:
the 12-bit shift is confirmed on every frame, the frame-type labels disagree in exactly one
direction (D18, D27), and D7's description of the primary validation dataset was wrong in a
way that turned out to be an improvement (D28).

It also closed a scheduled task without a bench session: D9's suspected dark gaps do not
exist (D32).

Next is **02**, the noise estimator — which the quantisation cell above already argues is not
free to choose.